# Kumatage with the apparent size of the Sun

The [kumatage tutorial](../kumatage.ipynb) treats the Sun as a point source: all
the sunlight comes from one direction $(\theta_s, \phi_s = 0)$. Seen from the
Earth, the Sun is a disk of about 0.53° in diameter, and each point of the disk
illuminates the sea from a slightly different direction. This notebook
replicates the kumatage of the tutorial with an extended, limb-darkened Sun:

* how the solar disk maps into Sun zenith and azimuth angles,
* how the glint of the disk is built from the glints of its points,
* when the apparent size of the Sun matters: Sun height, wind speed, distance
  to the image of the Sun.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm
from matplotlib.patches import Circle

from coxmunk import sunglint

rad = np.radians

## The solar disk in Sun angles

The apparent radius of the Sun is $r_\odot \approx 0.2665°$ (0.262° to 0.272°
between aphelion and perihelion). The disk is sampled with concentric rings of
points of about equal area, $1 + 6 + 12 = 19$ points for two rings. The disk is
brighter at its center than at its limb; with the linear limb-darkening law

$$\frac{B(\mu)}{B(1)} = 1 - u\,(1 - \mu), \qquad \mu = \sqrt{1 - (r/r_\odot)^2},$$

and $u \approx 0.6$ in the visible, each point receives a weight $w_j \propto 1 - u(1 - \mu_j)$,
with $\sum_j w_j = 1$.

A point of the disk at the angular offsets $(\Delta e, \Delta h)$ (vertical,
horizontal) from the center of the Sun has the direction

$$\mathbf{s}_j \propto \mathbf{s}_0 + \tan\Delta e\,\mathbf{e}_{up} + \tan\Delta h\,\mathbf{e}_{h},$$

with $\mathbf{s}_0 = (\cos e_0, 0, \sin e_0)$ the center of the Sun at the
elevation $e_0$, $\mathbf{e}_{up} = (-\sin e_0, 0, \cos e_0)$ and
$\mathbf{e}_h = (0, 1, 0)$. Its zenith angle and azimuth $(\theta_{s,j}, \phi_{s,j})$
follow. The elevation offsets go directly into the Sun zenith angle, while a
horizontal offset $\Delta h$ changes the azimuth by about $\Delta h / \sin\theta_s$:
negligible for a low Sun, but large for a Sun close to the zenith.

In [ ]:
R_SUN = 0.2665           # apparent radius of the Sun (deg)


def disk_samples(n_rings=2, radius=R_SUN, u=0.6):
    """Offsets (d_elevation, d_horizontal) in deg of points of the solar disk and their weights."""
    pts = [(0., 0.)]
    for k in range(1, n_rings + 1):
        r, n = radius * k / (n_rings + 0.5), 6 * k
        pts += [(r * np.cos(2 * np.pi * (m + 0.5 * (k % 2)) / n), r * np.sin(2 * np.pi * (m + 0.5 * (k % 2)) / n))
                for m in range(n)]
    pts = np.array(pts)
    mu = np.sqrt(np.clip(1 - (np.hypot(*pts.T) / radius) ** 2, 0, 1))
    w = 1 - u * (1 - mu)
    return pts, w / w.sum()


def sun_angles(sun_elevation, offsets):
    """Zenith angles and azimuths (deg) of the points of the disk; Sun center at azimuth 0."""
    e0 = rad(sun_elevation)
    s0 = np.array([np.cos(e0), 0., np.sin(e0)])
    e_up = np.array([-np.sin(e0), 0., np.cos(e0)])
    e_h = np.array([0., 1., 0.])
    s = s0 + np.tan(rad(offsets[:, :1])) * e_up + np.tan(rad(offsets[:, 1:])) * e_h
    s /= np.linalg.norm(s, axis=1, keepdims=True)
    return np.degrees(np.arccos(s[:, 2])), np.degrees(np.arctan2(s[:, 1], s[:, 0]))


offsets, weights = disk_samples()
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
sc = axs[0].scatter(offsets[:, 1], offsets[:, 0], c=weights, s=180, cmap='YlOrRd_r')
axs[0].add_patch(Circle((0, 0), R_SUN, fill=False, color='k'))
axs[0].set_aspect('equal')
axs[0].set_xlabel('horizontal offset (deg)')
axs[0].set_ylabel('vertical offset (deg)')
axs[0].set_title('disk samples (color: limb-darkened weight)')
fig.colorbar(sc, ax=axs[0])
for ax, e0 in zip(axs[1:], (10., 88.)):
    sza_j, phi_j = sun_angles(e0, offsets)
    ax.scatter(phi_j, sza_j, c=weights, s=120, cmap='YlOrRd_r')
    ax.invert_yaxis()
    ax.set_xlabel(r'Sun azimuth $\phi_{s,j}$ (deg)')
    ax.set_ylabel(r'Sun zenith angle $\theta_{s,j}$ (deg)')
    ax.set_title(f'Sun elevation {e0:.0f}°: azimuth spread ±{phi_j.max():.2f}°')
plt.tight_layout()

## Glint of the solar disk

Each point of the disk reflects on the sea like a point source of irradiance
$w_j E_0$ from $(\theta_{s,j}, \phi_{s,j})$. The radiances add up, so that the
reflectance of the disk, normalized like the point source by the irradiance
of the whole Sun $E_0\cos\theta_s$ at its center, is

$$\rho_\odot = \sum_j w_j\,\rho(\theta_{s,j}, \theta_v, \phi - \phi_{s,j})\,
\frac{\cos\theta_{s,j}}{\cos\theta_s}.$$

The relative azimuths of the sensor and of the wind are counted from each
point of the disk: the sensor and the wind keep their direction, the Sun
moves. The Stokes parameters Q and U are referenced to the meridian plane of
the sensor, which does not depend on the Sun position, so they add up like I.
Points of the disk below the horizon do not illuminate the sea.

In [ ]:
def glint_disk(sun_elevation, delta, alpha, ws, wazi=0, n_rings=2, stats='bh2006'):
    """Stokes vector [I, Q, U, V] of the glint of the solar disk seen from a ship deck,
    at the depression delta and the azimuth alpha from the Sun (deg)."""
    off, w = disk_samples(n_rings) if n_rings else (np.zeros((1, 2)), np.ones(1))
    sza_j, phi_j = sun_angles(sun_elevation, off)
    mu0 = np.cos(rad(90. - sun_elevation))
    S = np.zeros(4)
    for sza, phi, wj in zip(sza_j, phi_j, w):
        if sza >= 90.:
            continue                    # this part of the Sun has set
        g = sunglint(sza, 90. - delta, 180. + alpha - phi)
        S += wj * np.array(g.sunglint(ws, wazi - phi, stats=stats)) * np.cos(rad(sza)) / mu0
    return S


def deck_view(sun_elevation, ws, n_rings=2):
    S = np.empty((4, len(delta), len(alpha)))
    for i, d in enumerate(delta):
        for j, a in enumerate(alpha):
            S[:, i, j] = glint_disk(sun_elevation, d, a, ws, n_rings=n_rings)
    return S


print('point source:', glint_disk(10, 10, 0, 5, n_rings=0)[0])
print('solar disk:  ', glint_disk(10, 10, 0, 5)[0])

The number of rings is chosen by checking the convergence in the most
demanding case of this notebook, a Sun 1° above the horizon over a calm sea:

In [ ]:
for d in (0.5, 1., 2.):
    I = [glint_disk(1., d, 0., 1., n_rings=n)[0] for n in (0, 1, 2, 4)]
    print(f'delta = {d:3.1f} deg  disk/point: ' + '  '.join(f'{n} rings {x / I[0]:.4f}' for n, x in zip((1, 2, 4), I[1:])))

Two rings (19 points) are within about 1.5 % of four rings (61 points) in this
worst case, and much closer for a higher Sun; they are used below.

## Kumatage of a low Sun

The kumatage is computed as in the tutorial, for a Sun 3° above the horizon,
first with a point source and then with the limb-darkened disk (drawn to
scale). The third row is the ratio of the two.

In [ ]:
sun_elevation = 3.
delta = np.linspace(0.1, 10, 60)        # depression angle below the horizon (deg)
alpha = np.linspace(-15, 15, 61)        # azimuth offset from the Sun (deg)
winds = (1, 5, 12)

point = {ws: deck_view(sun_elevation, ws, n_rings=0) for ws in winds}
disk = {ws: deck_view(sun_elevation, ws) for ws in winds}

fig, axs = plt.subplots(3, len(winds), figsize=(18, 10.5), sharex=True, sharey=True, layout='constrained')
for k, ws in enumerate(winds):
    for row, (S, name) in enumerate(((point[ws], 'point Sun'), (disk[ws], 'solar disk'))):
        ax = axs[row, k]
        im = ax.pcolormesh(alpha, -delta, np.clip(S[0], 1e-6, None), norm=LogNorm(1e-1, 1e5),
                           cmap='inferno', shading='auto')
        ax.set_title(f'{name}, {ws} m/s')
        ax.set_facecolor('#4a6fa5')
    axs[2, k].set_facecolor('0.6')
    fig.colorbar(im, ax=axs[:2, k], label='glint reflectance I', shrink=0.6)
    glint_visible = point[ws][0] > 0.1    # lower bound of the color scale above
    ratio = np.where(glint_visible, disk[ws][0] / np.where(glint_visible, point[ws][0], 1), np.nan)
    im = axs[2, k].pcolormesh(alpha, -delta, ratio, norm=TwoSlopeNorm(1, 0.8, 1.2), cmap='RdBu_r', shading='auto')
    axs[2, k].set_title(f'solar disk / point Sun, {ws} m/s')
    fig.colorbar(im, ax=axs[2, k], label='ratio', shrink=0.9)
for ax in axs.ravel():
    ax.add_patch(Circle((0, sun_elevation), R_SUN, color='gold'))
    ax.axhline(0, color='w', lw=0.8)
    ax.set_ylim(-10, 5)
    ax.set_aspect('equal')
for ax in axs[:, 0]:
    ax.set_ylabel('elevation (deg)')
for ax in axs[-1]:
    ax.set_xlabel('azimuth from the Sun (deg)')
plt.show()

The glitter path looks the same: the disk is much smaller than the glitter.
Even without wind, the Cox-Munk slopes have a standard deviation of about 3.6°
(the constant term of the slope variance), so that the reflected directions
spread over several degrees, ten times the radius of the Sun. Averaging over
the disk smooths the glint (in gray, where the glint is negligible, the ratio
is not shown): it is slightly reduced in the core of the glitter path,
by about 5 % at the image of the Sun and up to 15 % near the horizon for a
light wind, and increased by 20 % or more on the faint edges of the path,
which the disk widens a little.

## Effect of the Sun height

Below the Sun ($\alpha = 0$), the ratio between the glint of the disk and that
of the point Sun is plotted against the depression angle normalized by the
Sun elevation (1: flat-sea image of the Sun). For the lowest Sun, 0.15°, the
lower part of the disk is below the horizon.

In [ ]:
x = np.geomspace(0.05, 5, 60)          # depression / Sun elevation
fig, axs = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, ws in zip(axs, (1, 5)):
    for e0 in (20, 10, 3, 1, 0.15):
        xe = x[x * e0 <= 60]               # the glint is negligible beyond 60° of depression
        r = [glint_disk(e0, xi * e0, 0, ws)[0] / glint_disk(e0, xi * e0, 0, ws, n_rings=0)[0] for xi in xe]
        ax.semilogx(xe, r, label=f'Sun elevation {e0}°')
    ax.axvline(1, color='0.5', ls=':')
    ax.axhline(1, color='k', lw=0.8)
    ax.set_xlabel('depression angle / Sun elevation (1: flat-sea image of the Sun)')
    ax.set_title(f'Below the Sun, {ws} m/s')
axs[0].set_ylabel('glint of the solar disk / point Sun')
axs[0].legend()
plt.tight_layout()

For a Sun higher than about 10°, the point-source approximation is accurate to
better than 1–2 % everywhere. It fails for a Sun within a few degrees of the
horizon: the Sun zenith angles of the disk then span grazing angles where the
glint changes very fast with the Sun elevation, through the shadowing of the
facets and the $1/\cos\theta$ factors, and the average over the disk is no
longer the value at its center. When the Sun is setting, the part of the disk
below the horizon no longer contributes and the glint drops further.

## Polarization

The degree of polarization of the disk is the ratio of the averaged
Stokes parameters. It only depends on the geometry and changes very little:

In [ ]:
for ws in winds:
    mask = point[ws][0] > 1
    dolp_p = np.hypot(point[ws][1], point[ws][2])[mask] / point[ws][0][mask]
    dolp_d = np.hypot(disk[ws][1], disk[ws][2])[mask] / disk[ws][0][mask]
    print(f'{ws:>2} m/s: max |DoLP(disk) - DoLP(point)| = {np.max(np.abs(dolp_d - dolp_p)):.4f}')

## Limitations

* The atmosphere is ignored. For a low Sun, the refraction lifts the Sun by
  about 0.5° at the horizon and flattens the disk vertically, and the
  extinction differs between the upper and lower limbs (their air masses
  differ). Both change the brightness distribution over the disk much more
  than the limb darkening does.
* The limb-darkening coefficient depends on the wavelength ($u \approx 0.6$ in the
  visible, smaller in the near infrared).
* The sea is flat on average (no curvature, see
  [the kumatage over a wind gradient](kumatage_wind_gradient.ipynb)), and the
  Cox-Munk statistics cannot describe a mirror-like sea. On a nearly flat sea,
  the image of the Sun is a reflected disk, which the slope variance floor of
  the statistics smears out.